<a href="https://colab.research.google.com/github/AllaYermilko/olist-analytics/blob/main/notebook/olist_predict2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3. Python: Simple Forecast and Recommendations

In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE=42

FILE_IN = "https://raw.githubusercontent.com/AllaYermilko/olist-analytics/refs/heads/main/data/main_sales_region.csv"

In [46]:
df = pd.read_csv(FILE_IN)
df.head()

,order_id,order_purchase_t,ym,customer_state,region,category_en,price,freight_value,payment_method,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10,SP,Southeast,housewares,29.99,8.72,credit_card,4.0
1,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10,SP,Southeast,housewares,29.99,8.72,voucher,4.0
2,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10,SP,Southeast,housewares,29.99,8.72,voucher,4.0
3,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07,BA,Northeast,perfumery,118.70,22.76,boleto,4.0
4,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08,GO,Central-West,auto,159.90,19.22,credit_card,5.0


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115723 entries, 0 to 115722
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   order_id          115723 non-null  object 
 1   order_purchase_t  115723 non-null  object 
 2   ym                115723 non-null  object 
 3   customer_state    115723 non-null  object 
 4   region            115723 non-null  object 
 5   category_en       114062 non-null  object 
 6   price             115723 non-null  float64
 7   freight_value     115723 non-null  float64
 8   payment_method    115720 non-null  object 
 9   review_score      114862 non-null  float64
dtypes: float64(3), object(7)
memory usage: 8.8+ MB


In [48]:
df.isna().sum()

,0
order_id,0
order_purchase_t,0
ym,0
customer_state,0
region,0
category_en,1661
price,0
freight_value,0
payment_method,3
review_score,861


In [49]:
df = df.sort_values('ym').reset_index(drop=True)
print("Month (all): ", len(df))
df.head()

Month (all):  115723


,order_id,order_purchase_t,ym,customer_state,region,category_en,price,freight_value,payment_method,review_score
0,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,2016-09,SP,Southeast,health_beauty,44.99,2.83,NaN,1.0
1,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,2016-09,SP,Southeast,health_beauty,44.99,2.83,NaN,1.0
2,bfbd0f9bdef84302105ad712db648a6c,2016-09-15 12:16:38,2016-09,SP,Southeast,health_beauty,44.99,2.83,NaN,1.0
3,6ef172eee30cfbfa01516ce2eb2ee68f,2016-10-07 11:37:12,2016-10,SP,Southeast,telephony,18.90,10.96,boleto,5.0
4,0c22166d9f7e5761e397affa5af438c7,2016-10-09 22:31:44,2016-10,CE,Northeast,health_beauty,90.00,26.89,credit_card,5.0


In [50]:
df_lr = df[(df.ym >= '2017-01') & (df.ym <= '2018-08')].reset_index(drop=True)
df_lr = df_lr[['ym', 'price']]
print("Month (Day) (in process):", len(df_lr))
df_lr.head()

Month (Day) (in process): 115385


,ym,price
0,2017-01,44.60
1,2017-01,16.90
2,2017-01,99.90
3,2017-01,21.99
4,2017-01,39.00


In [51]:
df_lr.isna().sum()

,0
ym,0
price,0


In [52]:
df_lr = df_lr.groupby('ym').agg(
    revenue=('price', 'sum')
).reset_index
df_lr.head()

AttributeError: 'function' object has no attribute 'head'

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(df_lr.ym, df_lr.revenue, marker="o")
plt.xticks(rotation=45, ha='right')
plt.title("Monthly revenue (stable period)")
plt.tight_layout()
plt.show()

In [ ]:
df_lr['t'] = range(len(df_lr))
df_lr.head()

In [ ]:
X = df_lr[['t']]
y = df_lr['revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

model = LinearRegression().fit(X_train, y_train)

next_index = len(df_lr)
y_pred = model.predict(X_test)

forecast = model.predict(pd.DataFrame({
    't': [next_index]
}))[0]
print(f"Predict revenue for next month: R${forecast:,.0f}")
print(f"Average growth for next month: R${model.coef_[0]:,.0f}")
print("-" * 15)
print(f"MAE: R${mean_absolute_error(y_test, y_pred):,.0f}")
print(f"RMSE: R${np.sqrt(mean_squared_error(y_test, y_pred)):,.0f}")


print(f"Predict revenue for next month: R${forecast:,.0f} +- {mean_absolute_error(y_test, y_pred):,.0f}")

In [ ]:
df_lr["month"] = df_lr['ym'].str[-2:]
season = df_lr.groupby("month")["revenue"].mean().round(0)
print("Average growth for number month: ")
print(season.sort_values(ascending=False))